In [1]:
import pandas as pd 
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length
from pyspark.sql.functions import explode, avg
import re
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [2]:

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

/opt/conda/lib/python3.7/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/09/28 14:50:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Set the option to display all columns
pd.set_option('display.max_columns', None)


In [4]:

#path = "/Users/acgid/Downloads/nba_stats/"
#Individual Statistics
file_1 = "Player Award Shares.csv" #Truth data and dependent variable
file_2 = "Player Totals.csv"
file_3 = "Player Per Game.csv"
file_4 = "Player Play By Play.csv"
file_5 = "Player Shooting.csv"
#Advanced Individual Statistics
file_6 = "Advanced.csv"
file_7 = "Per 36 Minutes.csv"
file_8 = "Per 100 Poss.csv"
#Teams Info & Stats
file_9 = "Team Totals.csv"
file_10 = "Team Stats Per Game.csv"
file_11 = "Team Stats Per 100 Poss.csv"
file_12 = "Opponent Stats Per Game.csv"
file_13 = "Opponent Stats Per 100 Poss.csv"

In [5]:
award_df = spark.read.csv(file_1, header=True, inferSchema=True)
plyr_tot_df = spark.read.csv(file_2, header=True, inferSchema=True)
ppg_df = spark.read.csv(file_3, header=True, inferSchema=True)
plyr_pbp_df = spark.read.csv(file_4, header=True, inferSchema=True)
plyr_shtg_df = spark.read.csv(file_5, header=True, inferSchema=True)
adv_df = spark.read.csv(file_6, header=True, inferSchema=True)
plyr_p36min_df = spark.read.csv(file_7, header=True, inferSchema=True)
plyr_p100ps_df = spark.read.csv(file_8, header=True, inferSchema=True)
tm_tot_df = spark.read.csv(file_9, header=True, inferSchema=True)
tm_pg_df = spark.read.csv(file_10, header=True, inferSchema=True)
tm_p100ps_df = spark.read.csv(file_11, header=True, inferSchema=True)
op_pg_df = spark.read.csv(file_12, header=True, inferSchema=True)
op_p100ps_df = spark.read.csv(file_13, header=True, inferSchema=True)

In [6]:
# filter for 1980 or higher
award_df_filtered = award_df.filter(award_df.season >= 1980)
print("This is award_df_filtered after filter:",award_df_filtered.count())

plyr_tot_df_filtered = plyr_tot_df.filter(plyr_tot_df.season >= 1980)
print("This is plyr_tot_df_filtered after filter:", plyr_tot_df_filtered.count())

ppg_df_filtered = ppg_df.filter(ppg_df.season >= 1980)
print("This is ppg_df_filtered after filter:", ppg_df_filtered.count())

plyr_pbp_df_filtered = plyr_pbp_df.filter(plyr_pbp_df.season >= 1980)
print("This is plyr_pbp_df_filtered after filter:", plyr_pbp_df_filtered.count())

plyr_shtg_df_filtered = plyr_shtg_df.filter(plyr_shtg_df.season >= 1980)
print("This is plyr_shtg_df_filtered after filter:", plyr_shtg_df_filtered.count())

adv_df_filtered = adv_df.filter(adv_df.season >= 1980)
print("This is adv_df_filtered after filter:", adv_df_filtered.count())

plyr_p36min_df_filtered = plyr_p36min_df.filter(plyr_p36min_df.season >= 1980)
print("This is plyr_p36min_df_filtered after filter:", plyr_p36min_df_filtered.count())

plyr_p100ps_df_filtered = plyr_p100ps_df.filter(plyr_p100ps_df.season >= 1980)
print("This is plyr_p100ps_df_filtered after filter:", plyr_p100ps_df_filtered.count())

tm_tot_df_filtered = tm_tot_df.filter(tm_tot_df.season >= 1980)
print("This is tm_tot_df_filtered after filter:", tm_tot_df_filtered.count())

tm_pg_df_filtered = tm_pg_df.filter(tm_pg_df.season >= 1980)
print("This is tm_pg_df_filtered after filter:", tm_pg_df_filtered.count())

tm_p100ps_df_filtered = tm_p100ps_df.filter(tm_p100ps_df.season >= 1980)
print("This is tm_p100ps_df_filtered after filter:", tm_p100ps_df_filtered.count())

op_pg_df_filtered = op_pg_df.filter(op_pg_df.season >= 1980)
print("This is op_pg_df_filtered after filter:", op_pg_df_filtered.count())

op_p100ps_df_filtered = op_p100ps_df.filter(op_p100ps_df.season >= 1980)
print("This is op_p100ps_df_filtered after filter:",op_p100ps_df_filtered.count())

This is award_df_filtered after filter: 2897
This is plyr_tot_df_filtered after filter: 24616
This is ppg_df_filtered after filter: 24616
This is plyr_pbp_df_filtered after filter: 17521
This is plyr_shtg_df_filtered after filter: 17521
This is adv_df_filtered after filter: 24616
This is plyr_p36min_df_filtered after filter: 24616
This is plyr_p100ps_df_filtered after filter: 24616
This is tm_tot_df_filtered after filter: 1330
This is tm_pg_df_filtered after filter: 1330
This is tm_p100ps_df_filtered after filter: 1284
This is op_pg_df_filtered after filter: 1330
This is op_p100ps_df_filtered after filter: 1284


In [7]:
award_df_filtered.dtypes

[('season', 'int'),
 ('award', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'int'),
 ('first', 'string'),
 ('pts_won', 'string'),
 ('pts_max', 'string'),
 ('share', 'string'),
 ('winner', 'boolean')]

In [8]:
plyr_tot_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'string'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp', 'string'),
 ('fg', 'int'),
 ('fga', 'int'),
 ('fg_percent', 'string'),
 ('x3p', 'string'),
 ('x3pa', 'string'),
 ('x3p_percent', 'string'),
 ('x2p', 'string'),
 ('x2pa', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),
 ('ft', 'int'),
 ('fta', 'int'),
 ('ft_percent', 'string'),
 ('orb', 'string'),
 ('drb', 'string'),
 ('trb', 'string'),
 ('ast', 'int'),
 ('stl', 'string'),
 ('blk', 'string'),
 ('tov', 'string'),
 ('pf', 'string'),
 ('pts', 'int'),
 ('trp_dbl', 'string')]

In [9]:
ppg_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'string'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp_per_game', 'string'),
 ('fg_per_game', 'double'),
 ('fga_per_game', 'double'),
 ('fg_percent', 'string'),
 ('x3p_per_game', 'string'),
 ('x3pa_per_game', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_game', 'string'),
 ('x2pa_per_game', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),
 ('ft_per_game', 'double'),
 ('fta_per_game', 'double'),
 ('ft_percent', 'string'),
 ('orb_per_game', 'string'),
 ('drb_per_game', 'string'),
 ('trb_per_game', 'string'),
 ('ast_per_game', 'double'),
 ('stl_per_game', 'string'),
 ('blk_per_game', 'string'),
 ('tov_per_game', 'string'),
 ('pf_per_game', 'string'),
 ('pts_per_game', 'double')]

In [10]:
plyr_pbp_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'int'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'int'),
 ('mp', 'int'),
 ('pg_percent', 'string'),
 ('sg_percent', 'string'),
 ('sf_percent', 'string'),
 ('pf_percent', 'string'),
 ('c_percent', 'string'),
 ('on_court_plus_minus_per_100_poss', 'string'),
 ('net_plus_minus_per_100_poss', 'string'),
 ('bad_pass_turnover', 'int'),
 ('lost_ball_turnover', 'int'),
 ('shooting_foul_committed', 'int'),
 ('offensive_foul_committed', 'int'),
 ('shooting_foul_drawn', 'int'),
 ('offensive_foul_drawn', 'string'),
 ('points_generated_by_assists', 'int'),
 ('and1', 'int'),
 ('fga_blocked', 'int')]

In [11]:
plyr_shtg_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'int'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'int'),
 ('mp', 'int'),
 ('fg_percent', 'string'),
 ('avg_dist_fga', 'string'),
 ('percent_fga_from_x2p_range', 'string'),
 ('percent_fga_from_x0_3_range', 'string'),
 ('percent_fga_from_x3_10_range', 'string'),
 ('percent_fga_from_x10_16_range', 'string'),
 ('percent_fga_from_x16_3p_range', 'string'),
 ('percent_fga_from_x3p_range', 'string'),
 ('fg_percent_from_x2p_range', 'string'),
 ('fg_percent_from_x0_3_range', 'string'),
 ('fg_percent_from_x3_10_range', 'string'),
 ('fg_percent_from_x10_16_range', 'string'),
 ('fg_percent_from_x16_3p_range', 'string'),
 ('fg_percent_from_x3p_range', 'string'),
 ('percent_assisted_x2p_fg', 'string'),
 ('percent_assisted_x3p_fg', 'string'),
 ('percent_dunks_of_fga', 'string'),
 ('num_of_dunks', 'int'),
 ('percent_corner_3s_of_3pa', 'string'),
 ('corner_3_point_percent', 'string'),


In [12]:
adv_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'string'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp', 'string'),
 ('per', 'string'),
 ('ts_percent', 'string'),
 ('x3p_ar', 'string'),
 ('f_tr', 'string'),
 ('orb_percent', 'string'),
 ('drb_percent', 'string'),
 ('trb_percent', 'string'),
 ('ast_percent', 'string'),
 ('stl_percent', 'string'),
 ('blk_percent', 'string'),
 ('tov_percent', 'string'),
 ('usg_percent', 'string'),
 ('ows', 'string'),
 ('dws', 'string'),
 ('ws', 'string'),
 ('ws_48', 'string'),
 ('obpm', 'string'),
 ('dbpm', 'string'),
 ('bpm', 'string'),
 ('vorp', 'string')]

In [13]:
plyr_p36min_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'string'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp', 'int'),
 ('fg_per_36_min', 'string'),
 ('fga_per_36_min', 'string'),
 ('fg_percent', 'string'),
 ('x3p_per_36_min', 'string'),
 ('x3pa_per_36_min', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_36_min', 'string'),
 ('x2pa_per_36_min', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),
 ('ft_per_36_min', 'string'),
 ('fta_per_36_min', 'string'),
 ('ft_percent', 'string'),
 ('orb_per_36_min', 'string'),
 ('drb_per_36_min', 'string'),
 ('trb_per_36_min', 'string'),
 ('ast_per_36_min', 'string'),
 ('stl_per_36_min', 'string'),
 ('blk_per_36_min', 'string'),
 ('tov_per_36_min', 'string'),
 ('pf_per_36_min', 'string'),
 ('pts_per_36_min', 'string')]

In [14]:
plyr_p100ps_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'int'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp', 'int'),
 ('fg_per_100_poss', 'string'),
 ('fga_per_100_poss', 'string'),
 ('fg_percent', 'string'),
 ('x3p_per_100_poss', 'string'),
 ('x3pa_per_100_poss', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_100_poss', 'string'),
 ('x2pa_per_100_poss', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),
 ('ft_per_100_poss', 'string'),
 ('fta_per_100_poss', 'string'),
 ('ft_percent', 'string'),
 ('orb_per_100_poss', 'string'),
 ('drb_per_100_poss', 'string'),
 ('trb_per_100_poss', 'string'),
 ('ast_per_100_poss', 'string'),
 ('stl_per_100_poss', 'string'),
 ('blk_per_100_poss', 'string'),
 ('tov_per_100_poss', 'string'),
 ('pf_per_100_poss', 'string'),
 ('pts_per_100_poss', 'string'),
 ('o_rtg', 'string'),
 ('d_rtg', 'string')]

In [15]:
tm_tot_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('abbreviation', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'string'),
 ('mp', 'string'),
 ('fg', 'string'),
 ('fga', 'string'),
 ('fg_percent', 'string'),
 ('x3p', 'string'),
 ('x3pa', 'string'),
 ('x3p_percent', 'string'),
 ('x2p', 'string'),
 ('x2pa', 'string'),
 ('x2p_percent', 'string'),
 ('ft', 'string'),
 ('fta', 'string'),
 ('ft_percent', 'string'),
 ('orb', 'string'),
 ('drb', 'string'),
 ('trb', 'string'),
 ('ast', 'string'),
 ('stl', 'string'),
 ('blk', 'string'),
 ('tov', 'string'),
 ('pf', 'string'),
 ('pts', 'string')]

In [16]:
tm_pg_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('abbreviation', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'string'),
 ('mp_per_game', 'string'),
 ('fg_per_game', 'string'),
 ('fga_per_game', 'string'),
 ('fg_percent', 'string'),
 ('x3p_per_game', 'string'),
 ('x3pa_per_game', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_game', 'string'),
 ('x2pa_per_game', 'string'),
 ('x2p_percent', 'string'),
 ('ft_per_game', 'string'),
 ('fta_per_game', 'string'),
 ('ft_percent', 'string'),
 ('orb_per_game', 'string'),
 ('drb_per_game', 'string'),
 ('trb_per_game', 'string'),
 ('ast_per_game', 'string'),
 ('stl_per_game', 'string'),
 ('blk_per_game', 'string'),
 ('tov_per_game', 'string'),
 ('pf_per_game', 'string'),
 ('pts_per_game', 'string')]

In [17]:
op_pg_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('abbreviation', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'string'),
 ('mp_per_game', 'string'),
 ('opp_fg_per_game', 'string'),
 ('opp_fga_per_game', 'string'),
 ('opp_fg_percent', 'string'),
 ('opp_x3p_per_game', 'string'),
 ('opp_x3pa_per_game', 'string'),
 ('opp_x3p_percent', 'string'),
 ('opp_x2p_per_game', 'string'),
 ('opp_x2pa_per_game', 'string'),
 ('opp_x2p_percent', 'string'),
 ('opp_ft_per_game', 'string'),
 ('opp_fta_per_game', 'string'),
 ('opp_ft_percent', 'string'),
 ('opp_orb_per_game', 'string'),
 ('opp_drb_per_game', 'string'),
 ('opp_trb_per_game', 'string'),
 ('opp_ast_per_game', 'string'),
 ('opp_stl_per_game', 'string'),
 ('opp_blk_per_game', 'string'),
 ('opp_tov_per_game', 'string'),
 ('opp_pf_per_game', 'string'),
 ('opp_pts_per_game', 'string')]

In [18]:
tm_p100ps_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('abbreviation', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'int'),
 ('mp', 'int'),
 ('fg_per_100_poss', 'double'),
 ('fga_per_100_poss', 'double'),
 ('fg_percent', 'double'),
 ('x3p_per_100_poss', 'string'),
 ('x3pa_per_100_poss', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_100_poss', 'double'),
 ('x2pa_per_100_poss', 'double'),
 ('x2p_percent', 'double'),
 ('ft_per_100_poss', 'double'),
 ('fta_per_100_poss', 'double'),
 ('ft_percent', 'double'),
 ('orb_per_100_poss', 'double'),
 ('drb_per_100_poss', 'double'),
 ('trb_per_100_poss', 'double'),
 ('ast_per_100_poss', 'double'),
 ('stl_per_100_poss', 'double'),
 ('blk_per_100_poss', 'double'),
 ('tov_per_100_poss', 'double'),
 ('pf_per_100_poss', 'double'),
 ('pts_per_100_poss', 'double')]

In [19]:
op_p100ps_df_filtered.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('abbreviation', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'int'),
 ('mp', 'int'),
 ('opp_fg_per_100_poss', 'double'),
 ('opp_fga_per_100_poss', 'double'),
 ('opp_fg_percent', 'double'),
 ('opp_x3p_per_100_poss', 'string'),
 ('opp_x3pa_per_100_poss', 'string'),
 ('opp_x3p_percent', 'string'),
 ('opp_x2p_per_100_poss', 'double'),
 ('opp_x2pa_per_100_poss', 'double'),
 ('opp_x2p_percent', 'double'),
 ('opp_ft_per_100_poss', 'double'),
 ('opp_fta_per_100_poss', 'double'),
 ('opp_ft_percent', 'double'),
 ('opp_orb_per_100_poss', 'double'),
 ('opp_drb_per_100_poss', 'double'),
 ('opp_trb_per_100_poss', 'double'),
 ('opp_ast_per_100_poss', 'double'),
 ('opp_stl_per_100_poss', 'double'),
 ('opp_blk_per_100_poss', 'double'),
 ('opp_tov_per_100_poss', 'double'),
 ('opp_pf_per_100_poss', 'double'),
 ('opp_pts_per_100_poss', 'double')]

In [20]:
temp1 = plyr_tot_df_filtered.join(ppg_df_filtered, 
                                  on=["season","lg","player", "player_id", "age", "team", "pos","g","gs"], 
                                  how="left")
print("this is after merge:",temp1.count())



this is after merge: 24616


In [21]:
temp1.show()

25/09/28 14:50:58 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+---+--------------------+---------+---+----+---+---+---+----+---+----+----------+---+----+-----------+---+----+-----------+------------+---+---+----------+---+---+---+---+---+---+---+---+----+-------+-----------+-----------+------------+----------+------------+-------------+-----------+------------+-------------+-----------+------------+-----------+------------+----------+------------+------------+------------+------------+------------+------------+------------+-----------+------------+
|season| lg|              player|player_id|age|team|pos|  g| gs|  mp| fg| fga|fg_percent|x3p|x3pa|x3p_percent|x2p|x2pa|x2p_percent|e_fg_percent| ft|fta|ft_percent|orb|drb|trb|ast|stl|blk|tov| pf| pts|trp_dbl|mp_per_game|fg_per_game|fga_per_game|fg_percent|x3p_per_game|x3pa_per_game|x3p_percent|x2p_per_game|x2pa_per_game|x2p_percent|e_fg_percent|ft_per_game|fta_per_game|ft_percent|orb_per_game|drb_per_game|trb_per_game|ast_per_game|stl_per_game|blk_per_game|tov_per_game|pf_per_game|pts_per_game|


In [22]:
temp1.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'string'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp', 'string'),
 ('fg', 'int'),
 ('fga', 'int'),
 ('fg_percent', 'string'),
 ('x3p', 'string'),
 ('x3pa', 'string'),
 ('x3p_percent', 'string'),
 ('x2p', 'string'),
 ('x2pa', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),
 ('ft', 'int'),
 ('fta', 'int'),
 ('ft_percent', 'string'),
 ('orb', 'string'),
 ('drb', 'string'),
 ('trb', 'string'),
 ('ast', 'int'),
 ('stl', 'string'),
 ('blk', 'string'),
 ('tov', 'string'),
 ('pf', 'string'),
 ('pts', 'int'),
 ('trp_dbl', 'string'),
 ('mp_per_game', 'string'),
 ('fg_per_game', 'double'),
 ('fga_per_game', 'double'),
 ('fg_percent', 'string'),
 ('x3p_per_game', 'string'),
 ('x3pa_per_game', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_game', 'string'),
 ('x2pa_per_game', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),

In [23]:
temp2 = plyr_pbp_df_filtered.join(plyr_shtg_df_filtered, 
                                  on=["season","lg","player", "player_id", "age", "team", "pos","g","gs","mp"], 
                                  how="left")

print("this is after merge:",temp2.count())



this is after merge: 17521


In [24]:
temp2.show()

+------+---+--------------------+---------+---+----+---+---+---+----+----------+----------+----------+----------+---------+--------------------------------+---------------------------+-----------------+------------------+-----------------------+------------------------+-------------------+--------------------+---------------------------+----+-----------+----------+------------+--------------------------+---------------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+-------------------------+--------------------------+---------------------------+----------------------------+----------------------------+-------------------------+-----------------------+-----------------------+--------------------+------------+------------------------+----------------------+--------------------+---------------+
|season| lg|              player|player_id|age|team|pos|  g| gs|  mp|pg_percent|sg_percent|sf_percent|pf_percent|c_p

In [25]:
temp2.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'int'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'int'),
 ('mp', 'int'),
 ('pg_percent', 'string'),
 ('sg_percent', 'string'),
 ('sf_percent', 'string'),
 ('pf_percent', 'string'),
 ('c_percent', 'string'),
 ('on_court_plus_minus_per_100_poss', 'string'),
 ('net_plus_minus_per_100_poss', 'string'),
 ('bad_pass_turnover', 'int'),
 ('lost_ball_turnover', 'int'),
 ('shooting_foul_committed', 'int'),
 ('offensive_foul_committed', 'int'),
 ('shooting_foul_drawn', 'int'),
 ('offensive_foul_drawn', 'string'),
 ('points_generated_by_assists', 'int'),
 ('and1', 'int'),
 ('fga_blocked', 'int'),
 ('fg_percent', 'string'),
 ('avg_dist_fga', 'string'),
 ('percent_fga_from_x2p_range', 'string'),
 ('percent_fga_from_x0_3_range', 'string'),
 ('percent_fga_from_x3_10_range', 'string'),
 ('percent_fga_from_x10_16_range', 'string'),
 ('percent_fga_from_x16_3p_range', 'string'),
 ('perce

In [26]:
temp3_a = adv_df_filtered.join(plyr_p36min_df_filtered, 
                                  on=["season","lg","player", "player_id", "age", "team", "pos","g","gs","mp"], 
                                  how="left")

temp3 = temp3_a.join(plyr_p100ps_df_filtered, 
                                  on=["season","lg","player", "player_id", "age", "team", "pos","g","gs","mp"], 
                                  how="left")


print("this is after merge:",temp3.count())


this is after merge: 24616


In [27]:
temp3.show()

+------+---+--------------------+---------+---+----+---+---+---+----+----+----------+------+-----+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+----+---+----+------+----+----+----+----+-------------+--------------+----------+--------------+---------------+-----------+--------------+---------------+-----------+------------+-------------+--------------+----------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+--------------+---------------+----------------+----------+----------------+-----------------+-----------+----------------+-----------------+-----------+------------+---------------+----------------+----------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+----------------+-----+-----+
|season| lg|              player|player_id|age|team|pos|  g| gs|  mp| per|ts_percent|x3p_ar|

In [28]:
temp3.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('player', 'string'),
 ('player_id', 'string'),
 ('age', 'string'),
 ('team', 'string'),
 ('pos', 'string'),
 ('g', 'int'),
 ('gs', 'string'),
 ('mp', 'string'),
 ('per', 'string'),
 ('ts_percent', 'string'),
 ('x3p_ar', 'string'),
 ('f_tr', 'string'),
 ('orb_percent', 'string'),
 ('drb_percent', 'string'),
 ('trb_percent', 'string'),
 ('ast_percent', 'string'),
 ('stl_percent', 'string'),
 ('blk_percent', 'string'),
 ('tov_percent', 'string'),
 ('usg_percent', 'string'),
 ('ows', 'string'),
 ('dws', 'string'),
 ('ws', 'string'),
 ('ws_48', 'string'),
 ('obpm', 'string'),
 ('dbpm', 'string'),
 ('bpm', 'string'),
 ('vorp', 'string'),
 ('fg_per_36_min', 'string'),
 ('fga_per_36_min', 'string'),
 ('fg_percent', 'string'),
 ('x3p_per_36_min', 'string'),
 ('x3pa_per_36_min', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_36_min', 'string'),
 ('x2pa_per_36_min', 'string'),
 ('x2p_percent', 'string'),
 ('e_fg_percent', 'string'),
 ('ft_per_36_min', '

In [29]:
temp4_a = tm_tot_df_filtered.join(tm_pg_df_filtered, 
                         on=["season","lg","team", "playoffs", "g"], 
                         how="left")

temp4 = temp4_a.join(op_pg_df_filtered, 
                         on=["season","lg","team", "playoffs", "g"], 
                         how="left")


print("this is after merge:",temp4.count())



this is after merge: 1330


In [30]:
temp4.show()

+------+---+--------------------+--------+---+------------+-----+----+----+----------+----+----+-----------+----+----+-----------+----+----+----------+----+----+----+----+---+---+----+----+----+------------+-----------+-----------+------------+----------+------------+-------------+-----------+------------+-------------+-----------+-----------+------------+----------+------------+------------+------------+------------+------------+------------+------------+-----------+------------+------------+-----------+---------------+----------------+--------------+----------------+-----------------+---------------+----------------+-----------------+---------------+---------------+----------------+--------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+----------------+
|season| lg|                team|playoffs|  g|abbreviation|   mp|  fg| fga|fg_percent| x3p|x3pa|x3p_percent| x2p|x2pa|x2p_percent|  ft| fta

In [31]:
temp4.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'string'),
 ('abbreviation', 'string'),
 ('mp', 'string'),
 ('fg', 'string'),
 ('fga', 'string'),
 ('fg_percent', 'string'),
 ('x3p', 'string'),
 ('x3pa', 'string'),
 ('x3p_percent', 'string'),
 ('x2p', 'string'),
 ('x2pa', 'string'),
 ('x2p_percent', 'string'),
 ('ft', 'string'),
 ('fta', 'string'),
 ('ft_percent', 'string'),
 ('orb', 'string'),
 ('drb', 'string'),
 ('trb', 'string'),
 ('ast', 'string'),
 ('stl', 'string'),
 ('blk', 'string'),
 ('tov', 'string'),
 ('pf', 'string'),
 ('pts', 'string'),
 ('abbreviation', 'string'),
 ('mp_per_game', 'string'),
 ('fg_per_game', 'string'),
 ('fga_per_game', 'string'),
 ('fg_percent', 'string'),
 ('x3p_per_game', 'string'),
 ('x3pa_per_game', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_game', 'string'),
 ('x2pa_per_game', 'string'),
 ('x2p_percent', 'string'),
 ('ft_per_game', 'string'),
 ('fta_per_game', 'string'),
 ('ft_percent', 'string'),


In [32]:
temp5 = tm_p100ps_df_filtered.join(op_p100ps_df_filtered, 
                         on=["season","lg", "team", "abbreviation", "playoffs","g","mp"], 
                         how="left")

print("this is after merge:",temp5.count())

this is after merge: 1284


In [33]:
temp5.show()

+------+---+--------------------+------------+--------+---+-----+---------------+----------------+----------+----------------+-----------------+-----------+----------------+-----------------+-----------+---------------+----------------+----------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+----------------+-------------------+--------------------+--------------+--------------------+---------------------+---------------+--------------------+---------------------+---------------+-------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+
|season| lg|                team|abbreviation|playoffs|  g|   mp|fg_per_100_poss|fga_per_100_poss|fg_percent|x3p_per_100_poss|x3pa_per_100_poss|x3p_percent|x2p_per_100_poss|x2pa_per_

In [34]:
temp5.dtypes

[('season', 'int'),
 ('lg', 'string'),
 ('team', 'string'),
 ('abbreviation', 'string'),
 ('playoffs', 'boolean'),
 ('g', 'int'),
 ('mp', 'int'),
 ('fg_per_100_poss', 'double'),
 ('fga_per_100_poss', 'double'),
 ('fg_percent', 'double'),
 ('x3p_per_100_poss', 'string'),
 ('x3pa_per_100_poss', 'string'),
 ('x3p_percent', 'string'),
 ('x2p_per_100_poss', 'double'),
 ('x2pa_per_100_poss', 'double'),
 ('x2p_percent', 'double'),
 ('ft_per_100_poss', 'double'),
 ('fta_per_100_poss', 'double'),
 ('ft_percent', 'double'),
 ('orb_per_100_poss', 'double'),
 ('drb_per_100_poss', 'double'),
 ('trb_per_100_poss', 'double'),
 ('ast_per_100_poss', 'double'),
 ('stl_per_100_poss', 'double'),
 ('blk_per_100_poss', 'double'),
 ('tov_per_100_poss', 'double'),
 ('pf_per_100_poss', 'double'),
 ('pts_per_100_poss', 'double'),
 ('opp_fg_per_100_poss', 'double'),
 ('opp_fga_per_100_poss', 'double'),
 ('opp_fg_percent', 'double'),
 ('opp_x3p_per_100_poss', 'string'),
 ('opp_x3pa_per_100_poss', 'string'),
 ('op

## Merge for Final dataset

Merge Temp1, temp2, temp3, temp4, temp5 togethere. Below is an ideas. Not decided how to go about that. 

In [ ]:
temp6_a = temp1.join(temp2,
                     on=["season","lg","player", "player_id", "age", "team", "pos","g"], 
                     how="left")
print("Join temp1 and temp 2:", temp6_a.count())

temp6_b = temp6_a.join(temp3, 
                        on=["season","lg","player", "player_id", "age", "team", "pos","g"], 
                        how="left")
print("Join temp6_a and temp3:", temp6_b.count())

temp6_c = temp6_b.join(temp4, 
                       on=["season","lg","team", "g"], 
                       how="left")
print("Join temp6_b and temp4:", temp6_c.count())

temp6 = temp6_c.join(temp5,
                     on=["season","lg", "team", "g"],
                     how="left")
print("Join temp6_c and temp5:", temp6.count())

In [ ]:
temp7 = temp6.join(award_df_filtered,
                   on=["season","player", "player_id", "age"],
                   how="left")

In [ ]:
temp7.show()